# Clinical Concept Annotation Benchmarking

This notebook demonstrates how to benchmark the `ClinicalConceptAnnotator` using synthetic annotation datasets.

## Overview

Annotation benchmarking evaluates how well a method can identify and rank relevant SNOMED CT concepts for clinical text inputs.

### Key Metrics:
- **Precision@K**: Fraction of top-K predicted concepts that are relevant
- **Recall@K**: Fraction of all relevant concepts found in top-K
- **F1@K**: Harmonic mean of Precision and Recall at K
- **MRR (Mean Reciprocal Rank)**: Average of reciprocal ranks of first relevant concept

### Data Format:
```python
{
    'text': str,           # Clinical text to annotate
    'gold_cuis': List[str]  # Ground truth CUIs
}
```

In [ ]:
# Import required modules
from snomed_methods.benchmarking.annotation import (
    evaluate_annotator,
    generate_annotation_dataset,
)

# For demo: import the annotator (requires actual data)
# from snomed_methods import ClinicalConceptAnnotator

## Generate Synthetic Annotation Dataset

We create synthetic clinical notes with disease-specific terminology and associated gold-standard CUIs.

In [ ]:
# Generate a small annotation dataset
dataset = generate_annotation_dataset(num_samples=50)

print(f"Dataset size: {len(dataset)}")
print("\nFirst sample:")
sample = dataset[0]
print(f"  Text preview: {sample['text'][:100]}...")
print(f"  Gold CUIs: {sample['gold_cuis']}")

## Create Mock Annotation Function

For demonstration, we create a simple annotator that returns mock results. Replace this with your actual annotator.

In [ ]:
# Example: Simple keyword-based mock annotation
def simple_annotator(text: str):
    """Mock annotator that returns synthetic results based on text content."""
    text_lower = text.lower()

    cui_mapping = {
        "diabetes": ["C001", "C002"],
        "hypertension": ["C010", "C011"],
        "depression": ["C020", "C021"],
        "migraine": ["C030"],
        "pneumonia": ["C040", "C041"],
        "asthma": ["C050", "C051"],
    }

    predicted_cuis = []
    for keyword, cuis in cui_mapping.items():
        if keyword in text_lower:
            predicted_cuis.extend(cuis)

    # Add some generic concepts as well
    predicted_cuis.extend(["C999", "C998"])

    return predicted_cuis[:10]  # Return top 10


# Test the mock annotator
test_text = dataset[0]["text"]
result = simple_annotator(test_text)
print(f"Predicted CUIs: {result}")

## Evaluate the Annotation Method

Run evaluation using different K values to assess performance at different ranking positions.

In [ ]:
# Evaluate on the full dataset
results = evaluate_annotator(
    annotator_func=simple_annotator,
    dataset=dataset,
    k_values=[1, 3, 5, 10],
)

print("\n=== Annotation Benchmarking Results ===")
for metric, value in results.items():
    if metric != "num_samples":
        print(f"{metric}: {value:.4f}")

print(f"\nTotal samples evaluated: {results['num_samples']}")

## Working with Real SNOMED Data

When actual SNOMED CT data is available, you can use the `ClinicalConceptAnnotator`:

In [ ]:
# Example: Using real ClinicalConceptAnnotator (requires UK data path)
# from snomed_methods import ClinicalConceptAnnotator

# annotator = ClinicalConceptAnnotator(
#     uk_path="/path/to/uk_sct2cl_42.2.0",
#     model_path="/path/to/model",  # Optional: SapBERT or similar
# )

# def real_annotator(text):
#     result = annotator.annotate(text, top_k=10)
#     return [c.concept_id for c in result.top_concepts]

# results_real = evaluate_annotator(real_annotator, dataset[:20])
# print(results_real)

## Load Pre-generated Datasets

Pre-generated datasets allow for reproducible benchmarking across runs.

In [ ]:
from snomed_methods.benchmarking.annotation import load_annotation_datasets

# Load all pre-generated datasets
datasets = load_annotation_datasets()

for name, data in datasets.items():
    print(f"{name}: {len(data)} samples")

# Use medium dataset for evaluation
medium_dataset = datasets["medium"]
print("\nMedium dataset evaluation (first 3 samples):")
for i, sample in enumerate(medium_dataset[:3]):
    print(f"  Sample {i+1}: {sample['text'][:60]}... -> {sample['gold_cuis']}")

## Compare Different K Values

Analyze how precision/recall change with different ranking positions.

In [ ]:
# Evaluate with multiple K values to see the trade-off
k_options = [1, 2, 3, 5, 10]

results_k = evaluate_annotator(
    annotator_func=simple_annotator,
    dataset=dataset[:30],
    k_values=k_options,
)

print("\nPerformance at different K values:")
print(f"{'K':<5} {'Precision':<12} {'Recall':<12} {'F1':<10}")
print("-" * 45)

for k in k_options:
    p = results_k.get(f"precision@{k}", 0)
    r = results_k.get(f"recall@{k}", 0)
    f1 = results_k.get(f"f1@{k}", 0)
    print(f"{k:<5} {p:<12.4f} {r:<12.4f} {f1:<10.4f}")